<a href="https://colab.research.google.com/github/rfaoktvian/Dicoding_FinalProject/blob/main/FractureNet_High_Sensitivity_X_Ray_Image_Classification_for_Orthopedic_Diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BUSINESS UNDERSTANDING


### ABOUT PROJECT

Diagnosis patah tulang (bone fracture) melalui citra X-ray adalah salah satu tugas paling umum namun krusial dalam radiologi. Di unit gawat darurat (UGD) yang sibuk, beban kerja radiolog yang tinggi dapat menyebabkan kelelahan, yang berisiko pada missed diagnosis—terutama pada kasus retakan rambut (hairline fractures) yang sangat tipis dan samar.

Proyek ini bertujuan untuk membangun sistem pendukung keputusan klinis (Clinical Decision Support System) berbasis Deep Learning yang mampu melakukan Skrining Otomatis: Mendeteksi keberadaan patah tulang pada berbagai wilayah tubuh (Multi-Region) secara instan.

Dataset:
Dataset yang digunakan adalah Fracture Multi-Region X-ray Steps yang mencakup berbagai anatomi tubuh (seperti tangan, kaki, panggul, dll). Berikut sumber dari dataset yang digunakan:
https://www.kaggle.com/datasets/bmadushanirodrigo/fracture-multi-region-x-ray-data .


### IMPORT LIBRARY

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras import layers,models,callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report
import os

### LOAD DATASET

In [2]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = '/content/drive/MyDrive/Dicoding/FinalProject/ProyeksiGambar/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification'
TRAIN_PATH = os.path.join(base_dir,'train')
VAL_PATH = os.path.join(base_dir,'val')
TEST_PATH = os.path.join(base_dir,'test')

Mounted at /content/drive


## EXPLORATORY DATA ANALYSIS

In [ ]:
train_fractured = len(os.listdir(os.path.join(TRAIN_PATH, 'fractured')))
train_normal = len(os.listdir(os.path.join(TRAIN_PATH, 'not fractured')))

data = {'Category': ['Fractured', 'Normal'], 'Count': [train_fractured, train_normal]}
df = pd.DataFrame(data)

plt.figure(figsize=(8, 5))
sns.barplot(x='Category', y='Count', hue='Category', data=df, palette='viridis',legend=False)
plt.title('Distribusi Kelas pada Training Set')
plt.show()

print(f"Total Fractured: {train_fractured}")
print(f"Total Normal: {train_normal}")

In [ ]:
from PIL import Image,ImageFile

sample_dir = os.path.join(TRAIN_PATH, 'fractured')
sample_files = os.listdir(sample_dir)[:50]

widths = []
heights = []

for file in sample_files:
    img = Image.open(os.path.join(sample_dir, file))
    widths.append(img.size[0])
    heights.append(img.size[1])

plt.figure(figsize=(8, 5))
plt.scatter(widths, heights, alpha=0.5)
plt.xlabel('Width')
plt.ylabel('Height')
plt.title('Sebaran Resolusi Asli Gambar X-Ray')
plt.show()

In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
def plot_raw_samples(path, title):
    files = os.listdir(path)[:5]
    plt.figure(figsize=(15, 5))
    for i, file in enumerate(files):
        img = Image.open(os.path.join(path, file))
        plt.subplot(1, 5, i+1)
        plt.imshow(img, cmap='gray')
        plt.title(title)
        plt.axis('off')
    plt.show()
plot_raw_samples(os.path.join(TRAIN_PATH, 'fractured'), "Fractured")
plot_raw_samples(os.path.join(TRAIN_PATH, 'not fractured'), "Normal")

## DATA PREPROCESSING & AUGMENTATION DATA

In [ ]:
def apply_clahe(img):
    img_uint8 = img.astype('uint8')
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    final_img = clahe.apply(img_uint8)
    return final_img.reshape(224, 224, 1).astype('float32')

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    preprocessing_function=apply_clahe
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale'
)

In [ ]:
val_test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale'
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale',
    shuffle=False
)

In [ ]:
def build_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 1)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model = build_model()
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('fracture_net_best.h5', save_best_only=True, monitor='val_loss')

callbacks_list = [early_stop, checkpoint]

In [ ]:
EPOCHS = 10
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks_list
)